In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService



QiskitRuntimeService.save_account(
    channel="ibm_cloud",
    token="9_AK16MQaBHKVFplG1ajedNDpaQ-GrX7h-Yg2gSR7D2X",
       instance="crn:v1:bluemix:public:quantum-computing:us-east:a/cd74bb3bd74f4112961f4a1709b478dc:04954a13-704c-4a02-a8ad-c5433b73357b::",
    overwrite=True
)

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Save your credentials locally
QiskitRuntimeService.save_account(
    channel="ibm_cloud", 
    token="9_AK16MQaBHKVFplG1ajedNDpaQ-GrX7h-Yg2gSR7D2X",
    overwrite=True
)


In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Automatically fetches the saved token
service = QiskitRuntimeService() 


In [ ]:
import math
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

# ==========================================================
# USER PARAMETERS
# ==========================================================

N_QUBITS = 4
MARKED_STATE = "1011"        # Must have length = N_QUBITS
SHOTS = 2048

USE_SIMULATOR = False  # Set to False to run on a real device

IBM_API_KEY = "9_AK16MQaBHKVFplG1ajedNDpaQ-GrX7h-Yg2gSR7D2X"

# ==========================================================
# VALIDATION
# ==========================================================

if len(MARKED_STATE) != N_QUBITS:
    raise ValueError("Length of MARKED_STATE must equal N_QUBITS.")

# Reverse because Qiskit uses little-endian ordering
TARGET = MARKED_STATE[::-1]

# ==========================================================
# ORACLE
# ==========================================================

def oracle(qc, target):

    n = len(target)

    # Convert target state into |111...1>
    for i, bit in enumerate(target):
        if bit == "0":
            qc.x(i)

    # Phase flip
    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)

    # Restore qubits
    for i, bit in enumerate(target):
        if bit == "0":
            qc.x(i)


# ==========================================================
# DIFFUSER
# ==========================================================

def diffuser(qc, n):

    qc.h(range(n))
    qc.x(range(n))

    qc.h(n - 1)
    qc.mcx(list(range(n - 1)), n - 1)
    qc.h(n - 1)

    qc.x(range(n))
    qc.h(range(n))


# ==========================================================
# BUILD GROVER CIRCUIT
# ==========================================================

qc = QuantumCircuit(N_QUBITS)

# Uniform superposition
qc.h(range(N_QUBITS))

# Optimal iterations
iterations = max(
    1,
    int((math.pi / 4) * math.sqrt(2 ** N_QUBITS))
)

print(f"Using {iterations} Grover iterations")

for _ in range(iterations):

    oracle(qc, TARGET)

    diffuser(qc, N_QUBITS)

qc.measure_all()

# ==========================================================
# CONNECT TO IBM QUANTUM
# ==========================================================

service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token=IBM_API_KEY
)

if USE_SIMULATOR:

    backend = service.least_busy(
        simulator=True
    )

else:

    backend = service.least_busy(
        operational=True,
        simulator=False
    )

print("Backend:", backend.name)

# ==========================================================
# TRANSPILATION
# ==========================================================

transpiled = transpile(
    qc,
    backend=backend,
    optimization_level=3
)

# ==========================================================
# EXECUTION
# ==========================================================

sampler = Sampler(mode=backend)

job = sampler.run(
    [transpiled],
    shots=SHOTS
)

print("Job ID:", job.job_id())

result = job.result()

counts = result[0].data.meas.get_counts()

# ==========================================================
# DISPLAY RESULTS
# ==========================================================

print("\nMeasurement Counts\n")

sorted_counts = sorted(
    counts.items(),
    key=lambda x: x[1],
    reverse=True
)

for state, count in sorted_counts:

    marker = ""

    if state == MARKED_STATE:
        marker = " <--- TARGET"

    print(f"{state} : {count}{marker}")

qiskit_runtime_service._discover_account:WARNING:2026-07-28 21:32:42,828: Loading account with the given token. A saved account will not be used.


Using 3 Grover iterations


qiskit_runtime_service.__init__:WARNING:2026-07-28 21:32:46,901: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.
qiskit_runtime_service.backends:WARNING:2026-07-28 21:32:47,573: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-07-28 21:32:51,252: Using instance: open-instance, plan: open


Backend: ibm_marrakesh
Job ID: d9kd5bbhdfks73cjgh8g
